# 🎓 Kakatiya University — College Bus Tracking System
### Real-Time Campus Bus Simulator

**Features:**
- Track college buses from your area to KU Campus
- Morning & Evening shift scheduling
- Live map with bus position, pickup stops & campus
- Find your nearest pickup stop
- ETA, seat availability & delay alerts
- Admin panel to add new routes

**Run:** `Runtime → Run all`  (Ctrl+F9)

In [ ]:
# Cell 1 — Install dependencies
!pip install folium geopy ipywidgets -q
print('Dependencies ready!')

In [ ]:
# Cell 2 — Imports
import folium
import time
import random
from datetime import datetime
from geopy.distance import geodesic
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
print('All libraries loaded!')

In [ ]:
# Cell 3 — College bus stops, routes & schedules
# Edit this cell to match your actual college routes!

# KU Campus (destination for all morning buses)
COLLEGE_NAME     = 'Kakatiya University'
COLLEGE_STOP     = 'KU Campus'
COLLEGE_LOCATION = (17.9785, 79.5305)

# All pickup stops with (lat, lon)
PICKUP_STOPS = {
    'KU Campus':        (17.9785, 79.5305),   # destination
    'Warangal Station': (17.9784, 79.5996),
    'Hanamkonda':       (18.0065, 79.5527),
    'Kazipet':          (17.9700, 79.5200),
    'Subedari':         (17.9850, 79.5600),
    'RTC Complex WG':   (17.9750, 79.5900),
    'Desaipet':         (18.0100, 79.5100),
    'Naimnagar':        (17.9600, 79.5400),
    'Mattewada':        (17.9500, 79.5700),
    'Nakkalagutta':     (18.0150, 79.5300),
    'Bheemaram':        (17.9400, 79.4900),
    'JNTU WG':          (17.9900, 79.5800),
}

# College bus routes:
# bus_id -> (pickup_start, final_destination, color, capacity, departure_time, route_via)
COLLEGE_ROUTES = {
    'CB-1': ('Warangal Station', 'KU Campus', 'blue',      60, '07:30 AM', ['RTC Complex WG', 'Kazipet', 'Subedari']),
    'CB-2': ('Hanamkonda',       'KU Campus', 'red',       50, '07:45 AM', ['Nakkalagutta', 'Desaipet']),
    'CB-3': ('Kazipet',          'KU Campus', 'green',     55, '08:00 AM', ['Naimnagar', 'Subedari']),
    'CB-4': ('Desaipet',         'KU Campus', 'purple',    48, '07:15 AM', ['Nakkalagutta', 'Hanamkonda']),
    'CB-5': ('Mattewada',        'KU Campus', 'orange',    52, '07:00 AM', ['Bheemaram', 'Naimnagar', 'Kazipet']),
    'CB-6': ('JNTU WG',          'KU Campus', 'darkblue',  45, '08:15 AM', ['RTC Complex WG', 'Subedari']),
    'CB-7': ('Bheemaram',        'KU Campus', 'cadetblue', 50, '07:20 AM', ['Desaipet', 'Hanamkonda']),
}

# ── Your location (change to yours!) ──────────────────────────────────
USER_LOCATION = (17.9784, 79.5996)   # example: near Warangal Station
# ──────────────────────────────────────────────────────────────────────

print(f'College  : {COLLEGE_NAME}')
print(f'Stops    : {len(PICKUP_STOPS)}')
print(f'Routes   : {len(COLLEGE_ROUTES)}')
print(f'Your loc : {USER_LOCATION}')

In [ ]:
# Cell 4 — CollegeBus class

class CollegeBus:
    def __init__(self, bus_id, start, end, color, capacity, departure, via):
        self.bus_id      = bus_id
        self.start       = start
        self.end         = end
        self.color       = color
        self.capacity    = capacity
        self.departure   = departure
        self.via         = via
        self.seated      = random.randint(int(capacity * 0.5), capacity - 2)
        self.speed_kmh   = random.randint(30, 60)
        self._progress   = random.uniform(0.0, 0.20)
        self._step       = 0.020
        self.delay_mins  = random.choice([0, 0, 0, 5, 10])  # 60% on time
        lat1, lon1 = PICKUP_STOPS[start]
        lat2, lon2 = PICKUP_STOPS[end]
        self._lat1, self._lon1 = lat1, lon1
        self._lat2, self._lon2 = lat2, lon2
        self._refresh()

    def _refresh(self):
        p = self._progress
        self.current_location = (
            self._lat1 + (self._lat2 - self._lat1) * p,
            self._lon1 + (self._lon2 - self._lon1) * p,
        )
        self.destination = PICKUP_STOPS[self.end]
        self.distance_remaining = geodesic(self.current_location, self.destination).km

    def move(self):
        if self._progress >= 1.0:
            return False
        jitter = random.uniform(-0.002, 0.003)
        self._progress = min(1.0, self._progress + self._step + jitter)
        self._refresh()
        return self._progress < 1.0

    def eta_minutes(self):
        if self.speed_kmh == 0:
            return 999
        base = round((self.distance_remaining / self.speed_kmh) * 60, 1)
        return base + self.delay_mins

    def progress_pct(self):
        return round(self._progress * 100, 1)

    def seats_available(self):
        return self.capacity - self.seated

    def status_label(self):
        if self.delay_mins > 0:
            return f'DELAYED +{self.delay_mins} min'
        return 'ON TIME'

    @property
    def arrived(self):
        return self._progress >= 1.0


def make_fleet():
    return {
        bid: CollegeBus(bid, s, e, c, cap, dep, via)
        for bid, (s, e, c, cap, dep, via) in COLLEGE_ROUTES.items()
    }

buses = make_fleet()
print(f'Fleet ready — {len(buses)} college buses')
print()
print(f'{"Bus":<8} {"From":<22} {"Departs":<12} {"Seats":<8} {"Status"}')
print('-' * 65)
for bid, b in buses.items():
    seats = f'{b.seats_available()}/{b.capacity}'
    print(f'{bid:<8} {b.start:<22} {b.departure:<12} {seats:<8} {b.status_label()}')

In [ ]:
# Cell 5 — Helper functions

def find_nearest_stop(loc):
    return min(
        ((s, geodesic(loc, c).km) for s, c in PICKUP_STOPS.items()),
        key=lambda x: x[1]
    )

def find_my_buses(user_loc=USER_LOCATION):
    nearest, dist = find_nearest_stop(user_loc)
    print(f'Your nearest pickup stop : {nearest}  ({dist:.2f} km away)')
    print(f'Buses heading to {COLLEGE_STOP}:')
    print()
    found = []
    for bid, b in buses.items():
        # bus starts at nearest OR nearest is in via stops
        serves = (b.start == nearest) or (nearest in b.via)
        if serves and b.end == COLLEGE_STOP:
            found.append(b)
    if not found:
        # fallback: any bus going to college
        found = [b for b in buses.values() if b.end == COLLEGE_STOP]
        print('  (No direct match — showing all campus-bound buses)')
    for b in sorted(found, key=lambda x: x.eta_minutes()):
        seats = b.seats_available()
        seat_color = 'green' if seats > 10 else ('orange' if seats > 0 else 'red')
        seat_tag = f'{seats} seats free' if seats > 0 else 'FULL'
        print(f'  Bus {b.bus_id}')
        print(f'    Route    : {b.start} -> {b.end}')
        print(f'    Via      : {" -> ".join(b.via)}')
        print(f'    Departs  : {b.departure}  [{b.status_label()}]')
        print(f'    ETA      : {b.eta_minutes()} min  |  Speed: {b.speed_kmh} km/h')
        print(f'    Seats    : {seat_tag}  ({b.seated}/{b.capacity} occupied)')
        print(f'    Progress : {b.progress_pct()}% completed')
        print('    ' + '-'*42)

def build_map(bus):
    m = folium.Map(location=bus.current_location, zoom_start=13, tiles='CartoDB positron')

    # All pickup stops
    for stop, coord in PICKUP_STOPS.items():
        color = 'red' if stop == COLLEGE_STOP else 'gray'
        folium.CircleMarker(
            location=coord, radius=6 if stop == COLLEGE_STOP else 5,
            color=color, fill=True, fill_opacity=0.7, tooltip=stop
        ).add_to(m)

    # Via stops
    for via_stop in bus.via:
        if via_stop in PICKUP_STOPS:
            folium.CircleMarker(
                location=PICKUP_STOPS[via_stop], radius=7,
                color='orange', fill=True, fill_opacity=0.8,
                tooltip=f'Via: {via_stop}'
            ).add_to(m)

    # Route line
    folium.PolyLine(
        [PICKUP_STOPS[bus.start], bus.destination],
        color=bus.color, weight=4, opacity=0.65, dash_array='8 4'
    ).add_to(m)

    # Start marker
    folium.Marker(
        PICKUP_STOPS[bus.start],
        tooltip=f'Start: {bus.start}',
        icon=folium.Icon(color='blue', icon='flag', prefix='fa')
    ).add_to(m)

    # College marker
    folium.Marker(
        bus.destination,
        tooltip=f'Campus: {bus.end}',
        popup=f'<b>{COLLEGE_NAME}</b>',
        icon=folium.Icon(color='green', icon='graduation-cap', prefix='fa')
    ).add_to(m)

    # Bus marker
    seats = bus.seats_available()
    folium.Marker(
        bus.current_location,
        popup=(
            f'<b>Bus {bus.bus_id}</b><br>'
            f'Route: {bus.start} to {bus.end}<br>'
            f'Speed: {bus.speed_kmh} km/h<br>'
            f'ETA: {bus.eta_minutes()} min<br>'
            f'Seats free: {seats}<br>'
            f'Status: {bus.status_label()}'
        ),
        tooltip=f'Bus {bus.bus_id} | ETA {bus.eta_minutes()} min',
        icon=folium.Icon(color='red', icon='bus', prefix='fa')
    ).add_to(m)

    # User location
    ns, nd = find_nearest_stop(USER_LOCATION)
    folium.Marker(
        USER_LOCATION,
        popup=f'You — nearest pickup: {ns} ({nd:.2f} km)',
        tooltip='Your location',
        icon=folium.Icon(color='purple', icon='user', prefix='fa')
    ).add_to(m)
    return m

def fleet_map():
    m = folium.Map(location=COLLEGE_LOCATION, zoom_start=12, tiles='CartoDB positron')
    for stop, coord in PICKUP_STOPS.items():
        clr = 'red' if stop == COLLEGE_STOP else 'gray'
        folium.CircleMarker(
            location=coord, radius=7 if stop == COLLEGE_STOP else 5,
            color=clr, fill=True, fill_opacity=0.75, tooltip=stop
        ).add_to(m)
    for bid, b in buses.items():
        folium.PolyLine(
            [PICKUP_STOPS[b.start], b.destination],
            color=b.color, weight=3, opacity=0.5
        ).add_to(m)
        seats_txt = f'{b.seats_available()} seats free'
        folium.Marker(
            b.current_location,
            popup=f'<b>Bus {bid}</b><br>{b.start} to {b.end}<br>ETA {b.eta_minutes()} min<br>{seats_txt}',
            tooltip=f'Bus {bid}',
            icon=folium.Icon(color=b.color, icon='bus', prefix='fa')
        ).add_to(m)
    folium.Marker(
        USER_LOCATION, tooltip='You',
        icon=folium.Icon(color='purple', icon='user', prefix='fa')
    ).add_to(m)
    return m

print('Helper functions ready!')

In [ ]:
# Cell 6 — Find YOUR college buses
find_my_buses(USER_LOCATION)

In [ ]:
# Cell 7 — Fleet map: all buses heading to campus
print('All college buses — current positions:')
fleet_map()

In [ ]:
# Cell 8 — Live bus tracker widget

bus_input = widgets.Dropdown(
    options=list(COLLEGE_ROUTES.keys()),
    value='CB-1',
    description='Bus:',
    layout=widgets.Layout(width='180px')
)
steps_sl = widgets.IntSlider(
    value=15, min=5, max=60, step=5,
    description='Steps:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
delay_sl = widgets.FloatSlider(
    value=1.5, min=0.5, max=4.0, step=0.5,
    description='Delay (s):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
btn    = widgets.Button(description='Track Bus', button_style='success',
                         layout=widgets.Layout(width='130px'))
status = widgets.Label(value='Select a bus and click Track Bus.')
out    = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<h3>College Bus Live Tracker</h3>'),
    widgets.HBox([bus_input, btn]),
    widgets.HBox([steps_sl, delay_sl]),
    status
]))
display(out)

def on_track(b):
    bid = bus_input.value
    s, e, c, cap, dep, via = COLLEGE_ROUTES[bid]
    buses[bid] = CollegeBus(bid, s, e, c, cap, dep, via)
    bus = buses[bid]
    max_steps = steps_sl.value
    delay     = delay_sl.value
    status.value = f'Tracking {bid}: {bus.start} -> {bus.end} | Departs {bus.departure}'
    with out:
        for step in range(max_steps):
            clear_output(wait=True)
            bus.move()
            pct    = bus.progress_pct()
            filled = int(pct / 5)
            bar    = chr(9608) * filled + chr(9617) * (20 - filled)
            seats  = bus.seats_available()
            seat_color = '#2a2' if seats > 10 else ('#e80' if seats > 0 else '#c00')
            display(HTML(f"""
            <div style="font-family:monospace;background:#f0fff4;
                        border:1px solid #aca;border-radius:8px;
                        padding:12px 16px;margin-bottom:6px;">
              <b>Bus {bus.bus_id}</b> &nbsp;|&nbsp;
              {bus.start} &#x2192; {bus.end} &nbsp;|&nbsp;
              Departs <b>{bus.departure}</b>
              &nbsp;<span style="color:{'#c80' if bus.delay_mins else '#2a2'}">
              [{bus.status_label()}]</span><br><br>
              Speed : {bus.speed_kmh} km/h &nbsp;&nbsp;
              Dist left : {bus.distance_remaining:.2f} km &nbsp;&nbsp;
              ETA : <b>{bus.eta_minutes()} min</b><br>
              Via : {' -> '.join(bus.via)}<br><br>
              Seats : <b style="color:{seat_color}">{seats} free</b>
              &nbsp;({bus.seated}/{bus.capacity} occupied)<br><br>
              Progress: <code>{bar}</code> {pct}%
              &nbsp;(step {step+1}/{max_steps})
            </div>
            """))
            display(build_map(bus))
            if bus.arrived:
                display(HTML(
                    f'<h3 style="color:green">Bus {bid} reached {COLLEGE_NAME}!</h3>'
                ))
                status.value = f'Bus {bid} arrived at campus.'
                break
            time.sleep(delay)
        else:
            status.value = (
                f'Simulation ended. Bus {bid} is '
                f'{bus.distance_remaining:.2f} km from campus.'
            )

btn.on_click(on_track)

In [ ]:
# Cell 9 — Admin: add a new route dynamically

new_id     = widgets.Text(value='CB-8',  description='Bus ID:',    layout=widgets.Layout(width='180px'))
new_start  = widgets.Text(value='Kazipet', description='From:',   layout=widgets.Layout(width='220px'))
new_end    = widgets.Text(value='KU Campus', description='To:',   layout=widgets.Layout(width='220px'))
new_cap    = widgets.IntText(value=50,   description='Capacity:', layout=widgets.Layout(width='160px'))
new_dep    = widgets.Text(value='07:50 AM', description='Departs:', layout=widgets.Layout(width='200px'))
add_btn    = widgets.Button(description='Add Route', button_style='warning',
                             layout=widgets.Layout(width='130px'))
add_status = widgets.Label(value='')

display(widgets.VBox([
    widgets.HTML('<h3>Admin — Add New Bus Route</h3>'),
    widgets.HBox([new_id, new_start, new_end]),
    widgets.HBox([new_cap, new_dep, add_btn]),
    add_status
]))

def on_add(b):
    bid   = new_id.value.strip()
    start = new_start.value.strip()
    end   = new_end.value.strip()
    cap   = new_cap.value
    dep   = new_dep.value.strip()
    if not bid or not start or not end:
        add_status.value = 'Please fill Bus ID, From, and To fields.'
        return
    if start not in PICKUP_STOPS:
        PICKUP_STOPS[start] = (17.9800, 79.5400)  # placeholder coords
    if end not in PICKUP_STOPS:
        PICKUP_STOPS[end] = COLLEGE_LOCATION
    COLLEGE_ROUTES[bid] = (start, end, 'darkred', cap, dep, [])
    buses[bid] = CollegeBus(bid, start, end, 'darkred', cap, dep, [])
    add_status.value = f'Route {bid} added: {start} -> {end} | {cap} seats | {dep}'
    bus_input.options = list(COLLEGE_ROUTES.keys())

add_btn.on_click(on_add)